# medi-LLaMA — DPO Fine-Tuning

**NLP Assignment 4 — IBA | Track 1, Option A: SFT → DPO**

This notebook runs 5 DPO trials on top of the best SFT checkpoint (Trial 5) and produces the final base → SFT → DPO comparison.

**Prerequisites (upload when prompted in Cell 3):**
- `best_checkpoint.zip` — SFT Trial 5 LoRA adapter (`trial_5/` folder)
- `Base-Model.zip` — DPO dataset + test prompts

**Runtime:** Google Colab T4 GPU (~68–155 min per trial, ~8 hrs total)

**Crash recovery:** Completed trials save to Google Drive automatically. Re-run from Cell 2 after a disconnect — done trials are skipped.

## Cell 1 — Install packages
Run once; kernel auto-restarts. Skip on re-run.

In [ ]:
import subprocess, sys, os

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "transformers>=4.41.0,<5.0.0",   # 4.40.2 conflicts with sentence-transformers
    "peft>=0.10.0",
    "trl>=0.8.6",
    "torchao>=0.16.0",               # required to avoid PEFT compatibility error
    "datasets", "accelerate", "bitsandbytes",
    "sacrebleu", "bert_score",
], check=True)

print("Restarting runtime...")
os.kill(os.getpid(), 9)

## Cell 2 — Mount Google Drive
Start here on re-run (skip Cell 1). All checkpoints and results save here.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/medi_llama_dpo'
HF_CACHE   = os.path.join(DRIVE_DIR, 'hf_cache')  # TinyLlama cached here (~2.2 GB)
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(HF_CACHE,  exist_ok=True)

# Point HuggingFace cache to Drive so TinyLlama isn't re-downloaded every session
os.environ['HF_HOME']            = HF_CACHE
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'

PROGRESS_FILE = os.path.join(DRIVE_DIR, 'dpo_progress.json')
print('Drive mounted.')
print(f'  Checkpoints : {DRIVE_DIR}')
print(f'  Model cache : {HF_CACHE}')

## Cell 3 — Imports and config

In [ ]:
import os, json, time, gc
import torch
import pandas as pd
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from trl import DPOTrainer, DPOConfig
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn

BASE_MODEL      = 'TinyLlama/TinyLlama_v1.1'
SFT_ADAPTER_DIR = '/content/trial_5'           # extracted from best_checkpoint.zip
DPO_DATA_DIR    = '/content/Base-Model/dpo'    # extracted from Base-Model.zip
TEST_PROMPTS_PATH = '/content/Base-Model/test_prompts.json'

SYSTEM_PROMPT = (
    'You are an experienced and knowledgeable medical professional. '
    'Provide clear, factual, and helpful medical information.'
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Config ready | device:', device)

## Cell 4 — Upload and extract files
Click **Choose Files** and select both zip archives.

In [ ]:
from google.colab import files
import zipfile

print('Upload best_checkpoint.zip and Base-Model.zip')
uploaded = files.upload()

for fname in uploaded:
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content')
    print(f'  Extracted {fname}')

# Verify expected files are present
checks = [
    ('/content/trial_5/adapter_config.json', 'SFT adapter'),
    ('/content/Base-Model/dpo/train',        'DPO train data'),
    ('/content/Base-Model/test_prompts.json','Test prompts'),
]
for path, label in checks:
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {status}  {label}')

## Cell 5 — Load tokenizer, test prompts, and DPO dataset

In [ ]:
# Tokenizer — TinyLlama's LlamaTokenizer (SentencePiece, vocab 32k)
# We hardcode the [INST] prompt format rather than using apply_chat_template
# to guarantee identical formatting across library versions.
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, trust_remote_code=True, force_download=True
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Test prompts (10 clinical questions + gold answers from ChatGPT GPT-4o)
with open(TEST_PROMPTS_PATH) as f:
    TEST_PROMPTS = json.load(f)
REFERENCES = [p['gold_answer'] for p in TEST_PROMPTS]
print(f'Loaded {len(TEST_PROMPTS)} test prompts')

# DPO dataset — reformat from Alpaca ### Instruction: format to [INST] format
raw_dpo = load_from_disk(DPO_DATA_DIR)

def format_dpo_row(row):
    """Extract instruction from Alpaca format and re-wrap in [INST] format."""
    raw_prompt = row['prompt']
    if '### Instruction:\n' in raw_prompt:
        instruction = raw_prompt.split('### Instruction:\n', 1)[1].strip()
    else:
        instruction = raw_prompt.strip()
    formatted_prompt = (
        f'<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n'
        f'{instruction} [/INST] '
    )
    return {
        'prompt':   formatted_prompt,
        'chosen':   str(row['chosen']).strip(),
        'rejected': str(row['rejected']).strip(),
    }

# Cap at 3000/300 in case dataset grows; currently 1951/217
n_train = min(3000, len(raw_dpo['train']))
n_val   = min(300,  len(raw_dpo['test']))
dpo_train = raw_dpo['train'].select(range(n_train)).map(format_dpo_row)
dpo_val   = raw_dpo['test'].select(range(n_val)).map(format_dpo_row)

# Drop identical chosen/rejected pairs (zero preference signal)
dpo_train = dpo_train.filter(lambda x: x['chosen'] != x['rejected'])
dpo_val   = dpo_val.filter(lambda x:   x['chosen'] != x['rejected'])

print(f'DPO train: {len(dpo_train)} triples | val: {len(dpo_val)} triples')

## Cell 6 — Evaluation helpers (BLEU + BERTScore)

In [ ]:
def generate_response(model, tokenizer, question, max_new_tokens=300):
    """Greedy decoding with the [INST] prompt format."""
    prompt = (
        f'<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n'
        f'{question} [/INST] '
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                         # greedy, deterministic
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def evaluate_model(model, tokenizer, label=''):
    """Run inference on all 10 test prompts and compute BLEU + BERTScore F1."""
    print(f'  Evaluating [{label}]...')
    preds = [generate_response(model, tokenizer, p['question']) for p in TEST_PROMPTS]

    bleu_metric = BLEU(effective_order=True)
    bleu_scores = [bleu_metric.sentence_score(p, [r]).score
                   for p, r in zip(preds, REFERENCES)]
    avg_bleu = round(sum(bleu_scores) / len(bleu_scores), 4)

    _, _, F1 = bert_score_fn(preds, REFERENCES, lang='en', verbose=False)
    avg_bert  = round(F1.mean().item(), 4)

    print(f'  ✅ BLEU: {avg_bleu}  |  BERTScore F1: {avg_bert}')
    return {'label': label, 'bleu': avg_bleu, 'bertscore_f1': avg_bert, 'predictions': preds}

## Cell 7 — Drive progress helpers
Saves/loads completed trial results to Drive so sessions can resume after crashes.

In [ ]:
def save_progress(results_list):
    safe = [{k: v for k, v in r.items() if k != 'predictions'} for r in results_list]
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(safe, f, indent=2)
    print(f'  Progress saved ({len(safe)} trial(s) done)')


def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            data = json.load(f)
        ids = [r['trial_id'] for r in data]
        print(f'  Resuming — completed trials: {ids}')
        return data
    print('  No saved progress — starting fresh')
    return []

## Cell 8 — DPO trial configurations

| Trial | β | LR | Batch | Epochs | What we're testing |
|-------|---|-----|-------|--------|--------------------|
| 1 | 0.10 | 5e-5 | 2 | 1 | Moderate baseline |
| 2 | 0.20 | 5e-5 | 2 | 1 | Higher β — tighter regularisation |
| 3 | 0.10 | 1e-4 | 2 | 1 | Same β, doubled LR |
| 4 | 0.30 | 5e-5 | 1 | 2 | Conservative β, extended training |
| 5 | 0.05 | 3e-5 | 2 | 1 | Low β — aggressive policy update |

**β** is the key DPO parameter: lower = bigger shift from the SFT reference distribution.

In [ ]:
DPO_TRIALS = [
    {'trial_id': 1, 'beta': 0.10, 'lr': 5e-5,  'batch_size': 2, 'epochs': 1},
    {'trial_id': 2, 'beta': 0.20, 'lr': 5e-5,  'batch_size': 2, 'epochs': 1},
    {'trial_id': 3, 'beta': 0.10, 'lr': 1e-4,  'batch_size': 2, 'epochs': 1},
    {'trial_id': 4, 'beta': 0.30, 'lr': 5e-5,  'batch_size': 1, 'epochs': 2},
    {'trial_id': 5, 'beta': 0.05, 'lr': 3e-5,  'batch_size': 2, 'epochs': 1},
]
print(f'✅ {len(DPO_TRIALS)} DPO trials defined')

## Cell 9 — DPO trial runner
Loads base + merges SFT adapter, adds fresh DPO LoRA, trains, evaluates, saves to Drive.

In [ ]:
def run_dpo_trial(cfg):
    gc.collect()
    torch.cuda.empty_cache()

    tid = cfg['trial_id']
    trial_dir = os.path.join(DRIVE_DIR, f'dpo_trial_{tid}')
    os.makedirs(trial_dir, exist_ok=True)

    print(f"\n{'='*65}")
    print(f" DPO TRIAL {tid}  |  beta={cfg['beta']}  lr={cfg['lr']}  "
          f"batch={cfg['batch_size']}  epochs={cfg['epochs']}")
    print(f" Checkpoint: {trial_dir}")
    print(f"{'='*65}")

    # 1. Load base model and bake SFT adapter into it
    # After merge_and_unload(), the reference pass (adapters OFF) falls back
    # to these SFT-merged weights — exactly the correct DPO reference.
    print('  Loading base + merging SFT adapter...')
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(model, SFT_ADAPTER_DIR)
    model = model.merge_and_unload()      # bakes SFT weights in permanently

    # 2. Add fresh LoRA adapters for DPO training
    # r=32 (lower than SFT's r=64) to keep VRAM manageable with adapter switching.
    dpo_lora = LoraConfig(
        r=32, lora_alpha=64,
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
        lora_dropout=0.05, bias='none',
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, dpo_lora)
    model.print_trainable_parameters()

    # 3. DPO training config
    # ref_model=None → DPOTrainer uses adapter switching:
    #   adapters ON  = trainable policy π_θ
    #   adapters OFF = frozen reference π_ref (the SFT-merged weights)
    dpo_args = DPOConfig(
        output_dir=trial_dir,
        num_train_epochs=cfg['epochs'],
        per_device_train_batch_size=cfg['batch_size'],
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,   # effective batch = cfg['batch_size'] * 4
        learning_rate=cfg['lr'],
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        bf16=True, fp16=False,
        beta=cfg['beta'],
        logging_steps=50,
        eval_strategy='steps',
        eval_steps=200,
        save_strategy='steps',
        save_steps=200,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        report_to='none',
    )

    trainer = DPOTrainer(
        model=model,
        ref_model=None,          # adapter switching — no second model loaded
        args=dpo_args,
        train_dataset=dpo_train,
        eval_dataset=dpo_val,
        processing_class=tokenizer,
    )

    # 4. Train
    t0 = time.time()
    trainer.train()
    train_time = round((time.time() - t0) / 60, 1)

    val_logs = [l for l in trainer.state.log_history if 'eval_loss' in l]
    val_loss = round(val_logs[-1]['eval_loss'], 4) if val_logs else None

    # 5. Save adapter to Drive
    trainer.save_model(trial_dir)
    print(f'  Saved to {trial_dir}')

    # 6. Evaluate on 10 test prompts
    eval_res = evaluate_model(model, tokenizer, label=f'DPO Trial {tid}')

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return {
        'trial_id': tid, 'config': cfg,
        'val_loss': val_loss, 'train_time_min': train_time,
        'bleu': eval_res['bleu'],
        'bertscore_f1': eval_res['bertscore_f1'],
        'predictions': eval_res['predictions'],
        'checkpoint_dir': trial_dir,
    }

## Cell 10 — Run all 5 DPO trials

⏰ ~68–155 min per trial on T4. Total ~8 hrs.
Completed trials are loaded from Drive and skipped automatically.

In [ ]:
dpo_all_results = load_progress()
completed_ids   = {r['trial_id'] for r in dpo_all_results}

for cfg in DPO_TRIALS:
    tid = cfg['trial_id']
    if tid in completed_ids:
        print(f'  Trial {tid} already done — skipping')
        continue

    result = run_dpo_trial(cfg)
    dpo_all_results.append(result)
    save_progress(dpo_all_results)

    print(f"  Trial {result['trial_id']} done: "
          f"BLEU {result['bleu']}  BERTScore {result['bertscore_f1']}  "
          f"val_loss {result['val_loss']}  ({result['train_time_min']} min)")

print('\nAll DPO trials complete!')

## Cell 11 — Select best DPO model

Selection criterion: BERTScore F1 (primary) → BLEU → lowest val loss (tiebreaker).

In [ ]:
ranked = sorted(
    dpo_all_results,
    key=lambda x: (x['bertscore_f1'], x['bleu'], -(x['val_loss'] or 99)),
    reverse=True,
)
best_dpo = ranked[0]
BEST_DPO_CHECKPOINT = best_dpo['checkpoint_dir']

print('\n' + '='*60)
print(f"  BEST DPO MODEL → Trial {best_dpo['trial_id']}")
print(f"  beta={best_dpo['config']['beta']}  lr={best_dpo['config']['lr']}  "
      f"batch={best_dpo['config']['batch_size']}  epochs={best_dpo['config']['epochs']}")
print(f"  BLEU {best_dpo['bleu']}  |  BERTScore {best_dpo['bertscore_f1']}  |  val_loss {best_dpo['val_loss']}")
print('='*60)

print('\nAll trials ranked:')
for i, r in enumerate(ranked):
    mark = ' ← BEST' if i == 0 else ''
    print(f"  {i+1}. Trial {r['trial_id']} | BLEU {r['bleu']} | "
          f"BERTScore {r['bertscore_f1']} | val_loss {r['val_loss']}{mark}")

## Cell 12 — Final comparison table (Base → SFT → DPO)

In [ ]:
# Numbers from Person 1 (baseline) and Person 2 (SFT)
BASELINE_BLEU = 1.1166;  BASELINE_BERT = 0.7788
SFT_BEST_BLEU = 3.3385;  SFT_BEST_BERT = 0.8292

rows = [
    {'Model': 'Base TinyLlama', 'BLEU': BASELINE_BLEU, 'BERTScore F1': BASELINE_BERT,
     'Val Loss': '-', 'Notes': 'No fine-tuning'},
    {'Model': 'SFT Trial 5',   'BLEU': SFT_BEST_BLEU, 'BERTScore F1': SFT_BEST_BERT,
     'Val Loss': 1.330, 'Notes': 'rank=64, lr=5e-5'},
]
for r in dpo_all_results:
    mark = ' ★' if r['trial_id'] == best_dpo['trial_id'] else ''
    rows.append({
        'Model':        f"DPO Trial {r['trial_id']}{mark}",
        'BLEU':         r['bleu'],
        'BERTScore F1': r['bertscore_f1'],
        'Val Loss':     r['val_loss'],
        'Notes':        f"beta={r['config']['beta']} lr={r['config']['lr']}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## Cell 13 — Qualitative comparison (Base → SFT → DPO)

Loads each model in turn and generates on 3 sample prompts. Each model is deleted from GPU after inference to avoid OOM.

In [ ]:
SAMPLE_IDS = [0, 3, 7]   # Q1 Chronic Disease, Q4 Pathophysiology, Q8 Critical Care
responses  = {i: {} for i in SAMPLE_IDS}

# ── Base model ──────────────────────────────────────────────────────
print('Loading base model...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True,
)
model.eval()
for i in SAMPLE_IDS:
    responses[i]['base'] = generate_response(model, tokenizer, TEST_PROMPTS[i]['question'])
    print(f'  ✅ Base — Prompt {i+1}')
del model; gc.collect(); torch.cuda.empty_cache()

# ── SFT model ───────────────────────────────────────────────────────
print('\nLoading SFT model...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True,
)
model = PeftModel.from_pretrained(model, SFT_ADAPTER_DIR)
model.eval()
for i in SAMPLE_IDS:
    responses[i]['sft'] = generate_response(model, tokenizer, TEST_PROMPTS[i]['question'])
    print(f'  ✅ SFT — Prompt {i+1}')
del model; gc.collect(); torch.cuda.empty_cache()

# ── DPO model ───────────────────────────────────────────────────────
# Critical: must load as base → merge SFT → load DPO adapter.
# Loading DPO adapter directly on raw base gives the wrong model.
print('\nLoading DPO model...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True,
)
model = PeftModel.from_pretrained(model, SFT_ADAPTER_DIR)
model = model.merge_and_unload()                        # bake SFT in
model = PeftModel.from_pretrained(model, BEST_DPO_CHECKPOINT)  # add DPO on top
model.eval()
for i in SAMPLE_IDS:
    responses[i]['dpo'] = generate_response(model, tokenizer, TEST_PROMPTS[i]['question'])
    print(f'  ✅ DPO — Prompt {i+1}')
del model; gc.collect(); torch.cuda.empty_cache()

# ── Print comparison ────────────────────────────────────────────────
for i in SAMPLE_IDS:
    print(f"\n{'='*70}")
    print(f"PROMPT {i+1} [{TEST_PROMPTS[i]['category']}]")
    print(f"Q: {TEST_PROMPTS[i]['question']}")
    print(f"\n📌 GOLD:\n{TEST_PROMPTS[i]['gold_answer'][:400]}")
    print(f"\n🔴 BASE:\n{responses[i]['base'][:400]}")
    print(f"\n🟡 SFT :\n{responses[i]['sft'][:400]}")
    print(f"\n🟢 DPO :\n{responses[i]['dpo'][:400]}")
    print(f"{'='*70}")

## Cell 14 — Save final results to Drive

In [ ]:
final_output = {
    'baseline':  {'bleu': BASELINE_BLEU, 'bertscore_f1': BASELINE_BERT},
    'sft_best':  {'trial_id': 5, 'bleu': SFT_BEST_BLEU, 'bertscore_f1': SFT_BEST_BERT},
    'dpo_trials': [
        {'trial_id': r['trial_id'], 'config': r['config'],
         'bleu': r['bleu'], 'bertscore_f1': r['bertscore_f1'],
         'val_loss': r['val_loss'], 'train_time_min': r['train_time_min']}
        for r in dpo_all_results
    ],
    'best_dpo_trial_id': best_dpo['trial_id'],
}

out_json = os.path.join(DRIVE_DIR, 'dpo_final_results.json')
out_csv  = os.path.join(DRIVE_DIR, 'dpo_results_summary.csv')

with open(out_json, 'w') as f:
    json.dump(final_output, f, indent=2)
df.to_csv(out_csv, index=False)

print(f'Saved to Drive:')
print(f'  {out_json}')
print(f'  {out_csv}')
print(f'  {DRIVE_DIR}/dpo_trial_*/   (adapter checkpoints)')